# modular-diffusion: the stage tree and its parts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.7.0/colab/tree.ipynb)

A stage tree is one shared training run that is forked into one copy per stretch of the noise level; after the fork every copy trains alone on its own stretch. This notebook measures the tree's two constants (when to fork, and how many copies when each copy gets more steps), the tree against the plain model at twice the compute and at equal size, and what parts add inside it (`geolip.modular_diffusion.experiments.stages`): class-gated parts born with the body, at the fork, or afterwards on the frozen tree; a tree whose body never reads the class and gets its content only through such parts; a class the tree never saw, added by one row of the class table, by a block, or by both, each trained alone with everything else frozen; and what such a part keeps when it is reused without training: on the tree trained further, on a sibling tree, on another run's tree, or as one part serving every stage. The list also holds the alternative of retraining the whole tree with the new class, so the cost and the reach of a single part can be set against it. Chain reads on averaged trees close the list.

1. install at a release tag and print the environment;
2. the data with its content hash, the finished runs this batch starts from (fetched from the Hub), a smoke run of the picture score and of the new paths, and the size of the job list;
3. the job list; each finished job is shipped to the Hub at once; a progress line a minute; the summary; then the machine is given back.

A stopped session resumes by running the cells again: finished jobs keep their markers under `OUT`, and on a new machine the jobs already on the Hub are fetched back first.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[data,hub,judge,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.7.0"
import os, sys, time, json, shutil, threading, subprocess
import torch, geolip.modular_diffusion as MD
from geolip.modular_diffusion import environment
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 7, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
environment.check(None)
print("card:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (choose a GPU runtime)", "| cpus:", os.cpu_count())
OUT = "/content/tree"                # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The data comes from the Hub as parquet files and becomes one npz of uint8 arrays with a content hash. The finished runs this batch starts from are fetched from the results repository: the shared run stopped at 4,000 steps with its trainer state, and the tree of three copies made from it (seeds 0 and 1). The smoke run trains the reference for 60 steps and scores it (the score is checked against a second implementation and the class reader on the test pictures), then runs every new path for a few steps and checks its exactness keys.

In [ ]:
# C2 - the data, the runs this batch starts from, a smoke run, the job list's size
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import data, score, linker
from geolip.modular_diffusion.beds import PicturesBed
from geolip.modular_diffusion.experiments import time_split, content, stages, chain_reads
NPZ = f"{OUT}/data/cifar10_32.npz"
REF = f"{OUT}/data/reference_inception.npz"     # the reference statistics: computed once here, read by every job
if not os.path.exists(NPZ):
    snapshot_download(repo_id="uoft-cs/cifar10", repo_type="dataset", allow_patterns=["plain_text/*.parquet"], local_dir=f"{OUT}/parquet")
    data.cifar_npz(parquet_dir=f"{OUT}/parquet/plain_text", out=NPZ)
EXPECTED_CONTENT_HASH = "be50ca799cbe6795ff13bca750f53f16c974b9f4903082fc06031b9719700016"
got = data.npz_content_hash(NPZ)
print("content hash", got, "| expected", EXPECTED_CONTENT_HASH or "(not filled)")
assert not EXPECTED_CONTENT_HASH or got == EXPECTED_CONTENT_HASH, "the npz is not the expected data"

REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" in C3 = nothing shipped)
BASES = [f"{v}_s{s}" for v in ("source", "copies_k3") for s in (0, 1)]   # the shared run at 4,000 steps and its tree
hub_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"modular_diffusion/stages/{t}/*" for t in BASES], local_dir=f"{OUT}/hub")
HUB = f"{hub_dir}/modular_diffusion/stages"
for s in (0, 1):
    assert os.path.exists(f"{HUB}/source_s{s}/state/state.json"), f"the shared run of seed {s} is not in the results repository"
    assert os.path.exists(f"{HUB}/copies_k3_s{s}/avg_tail100/staged.json"), f"the tree of seed {s} is not in the results repository"

SMOKE = f"{OUT}/smoke/plain"
smoke = content.main(variant="plain", seed=0, data=NPZ, out=SMOKE, device="cuda", scale=0.01, score=1, score_n=5000, ref=REF, reader=1)
assert "score" in smoke["score"], smoke["score"]
READER_CHECK = smoke["score"].get("reader_check")     # written by the job that computes the reference file: this one
print("class reader on the test pictures:", READER_CHECK, "| on the smoke model's samples:", (smoke["score"].get("class_read") or {}).get("accuracy"))
from torchmetrics.image.fid import FrechetInceptionDistance
bed = PicturesBed(NPZ, "cuda")
model = linker.link(f"{SMOKE}/model", device="cuda")
fake = score.sample_pictures(model, bed, 5000, steps=50, batch=1000, device="cuda")
real = bed.train_pictures()
fid = FrechetInceptionDistance(feature=2048, antialias=False).to("cuda")     # the same resize as the library's judge
for i in range(0, real.shape[0], 1000):
    fid.update(score.to_pictures(real[i:i + 1000], bed).to("cuda"), real=True)
for i in range(0, fake.shape[0], 1000):
    fid.update(fake[i:i + 1000].to("cuda"), real=False)
other = float(fid.compute())
mine = smoke["score"]["score"]
print(f"picture score: library {mine:.4f} | torchmetrics, the same resize {other:.4f} (relative difference {abs(mine - other) / other:.2e}) (5,000 pictures)")
del model, bed, real, fake, fid
torch.cuda.empty_cache()

# the new paths for a few steps each, with det=on as the job list runs them
S = f"{OUT}/smoke"
common = dict(seed=0, data=NPZ, device="cuda", scale=0.01, det="on", score=0)
K3 = dict(cuts="0.25,0.65")
stages.main(variant="source", out=f"{S}/src", **common)
stages.main(variant="copies", out=f"{S}/tree", source=f"{S}/src", **K3, **common)
sm_fork = {p: stages.main(variant="copies", out=f"{S}/fork_{p}", source=f"{S}/src", parts=p, **K3, **common) for p in ("experts10", "shared1")}
stages.main(variant="source", out=f"{S}/bsrc", class_window="blind", parts="experts10", **common)
sm_blind = stages.main(variant="copies", out=f"{S}/btree", source=f"{S}/bsrc", **K3, **common)
stages.main(variant="source", out=f"{S}/nsrc", without_classes="9", **common)
stages.main(variant="copies", out=f"{S}/ntree", source=f"{S}/nsrc", **K3, **common)
stages.main(variant="copies", out=f"{S}/nsib", source=f"{S}/nsrc", stream=1, **K3, **common)
sm_after = {p: stages.main(variant="parts", out=f"{S}/after_{p}", base=f"{S}/tree", parts=p, total=2000, **common) for p in ("experts10", "shared1")}
sm_add = {k: stages.main(variant="parts", out=f"{S}/add_{k}", base=f"{S}/ntree", parts=p, only_classes="9", total=1500, **common)
          for k, p in (("row", "row:9"), ("block", "expert:9"), ("both", "row+expert:9"))}
sm_same = stages.main(variant="parts", out=f"{S}/graft_same", base=f"{S}/ntree", parts="row:9", init_from=f"{S}/add_row", total=0, **common)
sm_graft = stages.main(variant="parts", out=f"{S}/graft_sib", base=f"{S}/nsib", parts="row:9", init_from=f"{S}/add_row", total=0, **common)
sm_refit = stages.main(variant="parts", out=f"{S}/refit", base=f"{S}/nsib", parts="expert:9", init_from=f"{S}/add_block", only_classes="9", total=300, **common)
stages.main(variant="copies", out=f"{S}/none", source=f"{S}/nsrc", **common)                      # the same run with no fork: a tree of one
stages.main(variant="parts", out=f"{S}/add_row_one", base=f"{S}/none", parts="row:9", only_classes="9", total=1500, **common)
sm_all = stages.main(variant="parts", out=f"{S}/row_one_on_tree", base=f"{S}/ntree", parts="row:9", init_from=f"{S}/add_row_one", init_stage=0, total=0, **common)
stages.main(variant="copies", out=f"{S}/nopen", source=f"{S}/nsrc", without_classes="none", **K3, **common)   # the copies admitted to the class
for name, r in list(sm_after.items()) + list(sm_add.items()) + [("graft_same", sm_same), ("graft_sib", sm_graft), ("refit", sm_refit), ("one row for every stage", sm_all)]:
    assert r["as_source"]["equals_stored"], f"{name}: the base read again is not its stored read"
    assert r["exact"]["others_max_abs"] == 0.0, f"{name}: rows the new parts are not on have changed"
    assert r["parts_off"]["equals_base"], f"{name}: the tree without the new parts is not the base"
g8 = lambda r: r["averaged"]["g8"]["loss_over_levels"]
assert g8(sm_same) == g8(sm_add["row"]), "a part placed back on its own parent does not read as it did"
assert all(sm_same["graft"]["parents_equal"]) and not all(sm_graft["graft"]["parents_equal"])
chain_reads.main(variant="twin", base=f"{S}/tree/avg_tail100", donor=f"{S}/nsib/avg_tail100", stage=1, data=NPZ, out=f"{S}/read_twin", device="cuda", score=0, det="on")
time_split.main(variant="plain_wide", out=f"{S}/wide", avg="g8,tail100", **common)
print("smoke of the new paths: parts at the fork", {k: round(r["averaged"]["tail100"]["loss_over_levels"], 4) for k, r in sm_fork.items()},
      "| a blind tree born with its parts", round(sm_blind["averaged"]["tail100"]["loss_over_levels"], 4),
      "| parts after", {k: round(g8(r), 4) for k, r in sm_after.items()},
      "| a class added", {k: round(g8(r), 4) for k, r in sm_add.items()})
INSTRUMENTS = {"reader_check": READER_CHECK, "smoke_class_read": smoke["score"].get("class_read"),
               "score_check": {"library": mine, "second_same_resize": other, "pictures": 5000}}

# the job list: (tag, experiment, arguments, the jobs it waits for, its cost in reference-model steps)
SCORE = 1700                                   # one scoring of 10,000 pictures, in reference-model steps
RUNS = f"{OUT}/runs"
TREE = "score_last=0 score_avg=tail100"        # a tree is judged and scored on the average of its last 100 steps
K3, K5 = "cuts=0.25,0.65", "cuts=0.105,0.288,0.552,0.827"
J = []
for s in (0, 1):
    o = 1 - s
    src4k, tree4k = f"{HUB}/source_s{s}", f"{HUB}/copies_k3_s{s}"
    run = lambda t: f"{RUNS}/{t}_s{s}"
    # when to fork: the shared run stopped at 500 / 2,000 / 5,000 steps, three copies for the rest of 6,000 steps
    for at in (500, 2000, 5000):
        J += [(f"src_{at}_s{s}", "stages", f"variant=source seed={s} total={at} score=0", [], at),
              (f"tree_f{at}_s{s}", "stages", f"variant=copies seed={s} {K3} total={6000 - at} source={run(f'src_{at}')} {TREE}",
               [f"src_{at}_s{s}"], 6000 - at + SCORE)]
    # twice the compute: three copies or five on a shared run of 6,000 steps; the plain model at 12,000 steps; the plain
    # model at three times the size
    J += [(f"src_6000_s{s}", "stages", f"variant=source seed={s} total=6000 score=0", [], 6000),
          (f"tree3_x2_s{s}", "stages", f"variant=copies seed={s} {K3} total=6000 source={run('src_6000')} {TREE}", [f"src_6000_s{s}"], 6000 + SCORE),
          (f"tree5_x2_s{s}", "stages", f"variant=copies seed={s} {K5} total=6000 source={run('src_6000')} {TREE}", [f"src_6000_s{s}"], 6000 + SCORE),
          (f"plain_12k_s{s}", "stages", f"variant=source seed={s} total=12000 avg=g8,tail100 score_last=0", [], 12000 + 2 * SCORE),
          (f"plain_wide_avg_s{s}", "time_split", f"variant=plain_wide seed={s} avg=g8,tail100 score_last=0", [], 12500 + 2 * SCORE)]
    # parts in the tree forked at 4,000: born after (on the frozen tree), at the fork, at the start; class-gated parts
    # and one shared part of the same kind
    J += [(f"after_exp_s{s}", "stages", f"variant=parts seed={s} base={tree4k} parts=experts10 total=2000", [], 2200 + SCORE),
          (f"after_shared_s{s}", "stages", f"variant=parts seed={s} base={tree4k} parts=shared1 total=2000", [], 2200 + SCORE),
          (f"fork_exp_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={src4k} parts=experts10 {TREE}", [], 2400 + SCORE),
          (f"fork_shared_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={src4k} parts=shared1 {TREE}", [], 2400 + SCORE),
          (f"start_src_exp_s{s}", "stages", f"variant=source seed={s} total=4000 parts=experts10 score=0", [], 4700),
          (f"start_exp_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('start_src_exp')} {TREE}", [f"start_src_exp_s{s}"], 2400 + SCORE)]
    # a body that never reads the class: its tree, class-gated parts born at the fork, and born with the body
    J += [(f"blind_src_s{s}", "stages", f"variant=source seed={s} total=4000 class_window=blind score=0", [], 4000),
          (f"blind_tree_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('blind_src')} {TREE}", [f"blind_src_s{s}"], 2000 + SCORE),
          (f"blind_fork_exp_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('blind_src')} parts=experts10 {TREE}", [f"blind_src_s{s}"], 2400 + SCORE),
          (f"blind_start_src_exp_s{s}", "stages", f"variant=source seed={s} total=4000 class_window=blind parts=experts10 score=0", [], 4700),
          (f"blind_start_exp_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('blind_start_src_exp')} {TREE}", [f"blind_start_src_exp_s{s}"], 2400 + SCORE)]
    # a class the tree never saw: the tree trained without it, a sibling tree on other batches, the class added by a
    # row of the class table, by a block, by both; the parts placed on the sibling and on the other seed's tree
    add = f"only_classes=9 total=1500 base={run('nine_tree')}"
    J += [(f"nine_src_s{s}", "stages", f"variant=source seed={s} total=4000 without_classes=9 score=0", [], 4000),
          (f"nine_tree_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('nine_src')} {TREE}", [f"nine_src_s{s}"], 2000 + SCORE),
          (f"nine_tree_sib_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('nine_src')} stream=1 {TREE}", [f"nine_src_s{s}"], 2000 + SCORE),
          (f"add9_row_s{s}", "stages", f"variant=parts seed={s} parts=row:9 {add}", [f"nine_tree_s{s}"], 1200 + SCORE),
          (f"add9_block_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 {add}", [f"nine_tree_s{s}"], 1300 + SCORE),
          (f"add9_both_s{s}", "stages", f"variant=parts seed={s} parts=row+expert:9 {add}", [f"nine_tree_s{s}"], 1300 + SCORE)]
    for k, p in (("row", "row:9"), ("block", "expert:9")):
        J += [(f"{k}_on_sib_s{s}", "stages", f"variant=parts seed={s} parts={p} base={run('nine_tree_sib')} init_from={run(f'add9_{k}')} total=0",
               [f"nine_tree_sib_s{s}", f"add9_{k}_s{s}"], 200 + SCORE),
              (f"{k}_on_other_s{s}", "stages", f"variant=parts seed={s} parts={p} base={RUNS}/nine_tree_s{o} init_from={run(f'add9_{k}')} total=0",
               [f"nine_tree_s{o}", f"add9_{k}_s{s}"], 200 + SCORE)]
    J += [(f"block_refit_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 base={run('nine_tree_sib')} init_from={run('add9_block')} only_classes=9 total=300",
           [f"nine_tree_sib_s{s}", f"add9_block_s{s}"], 500 + SCORE)]
    # reuse: the tree trained a quarter longer with its parts left as they are; one row trained once on the model
    # that was never forked and used by all three copies; the first stage's row used by every stage; and the
    # whole-tree alternative to adding a piece (the copies admitted to the class, everything trained)
    J += [(f"nine_tree_more_s{s}", "stages", f"variant=copies seed={s} {K3} total=2500 source={run('nine_src')} {TREE}", [f"nine_src_s{s}"], 2500 + SCORE),
          (f"nine_one_s{s}", "stages", f"variant=copies seed={s} total=2000 source={run('nine_src')} {TREE}", [f"nine_src_s{s}"], 2000 + SCORE),
          (f"add9_row_one_s{s}", "stages", f"variant=parts seed={s} parts=row:9 only_classes=9 total=1500 base={run('nine_one')}", [f"nine_one_s{s}"], 1500 + SCORE),
          (f"row_one_on_tree_s{s}", "stages", f"variant=parts seed={s} parts=row:9 base={run('nine_tree')} init_from={run('add9_row_one')} init_stage=0 total=0",
           [f"nine_tree_s{s}", f"add9_row_one_s{s}"], 200 + SCORE),
          (f"row_s0_all_s{s}", "stages", f"variant=parts seed={s} parts=row:9 base={run('nine_tree')} init_from={run('add9_row')} init_stage=0 total=0",
           [f"nine_tree_s{s}", f"add9_row_s{s}"], 200 + SCORE),
          (f"nine_tree_open_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('nine_src')} without_classes=none {TREE}", [f"nine_src_s{s}"], 2000 + SCORE)]
    for k, p in (("row", "row:9"), ("block", "expert:9")):
        J += [(f"{k}_on_more_s{s}", "stages", f"variant=parts seed={s} parts={p} base={run('nine_tree_more')} init_from={run(f'add9_{k}')} total=0",
               [f"nine_tree_more_s{s}", f"add9_{k}_s{s}"], 200 + SCORE)]
    # chain reads on the averaged tree forked at 4,000: one stage's copy from the other seed's tree, a hard switch,
    # sampling steps spread evenly or with half of them in the first quarter of the range
    mine_, other_ = f"{tree4k}/avg_tail100", f"{HUB}/copies_k3_s{o}/avg_tail100"
    J += [(f"read_twin_{k}_s{s}", "chain_reads", f"variant=twin base={mine_} donor={other_} stage={k} seed_label={s}", [], SCORE) for k in (0, 1, 2)]
    J += [(f"read_hard_s{s}", "chain_reads", f"variant=blend base={mine_} soft=0 seed_label={s}", [], SCORE)]
    J += [(f"read_steps_{plan}_{n}_s{s}", "chain_reads", f"variant=steps base={mine_} plan={plan} n={n} seed_label={s}", [], SCORE)
          for plan in ("uniform", "start") for n in (20, 50)]
ALL = J
assert len({t for t, *_ in ALL}) == len(ALL), "a job tag is listed twice"
STEPS = {t: c + 900 for t, _, _, _, c in ALL}      # a job's time in reference-model steps (its training, its reads, its scoring)
RATE = 36.0                                        # the reference model's steps per second on this card type with det=on
total = sum(STEPS.values())
print(f"the job list: {len(ALL)} jobs, {total:,.0f} reference-model steps in all: about {total / RATE / 60:.0f} min")

One runner works the job list in order, every job with deterministic algorithms switched on (`det=on`), so a job repeats to the last digit. Each finished job folder is shipped to `REPO` under its experiment's own folder with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). When the list has ended and its report is shipped, the machine is given back (`RELEASE`), since an idle machine keeps spending.

In [ ]:
# C3 - the runner, shipping after each job, a progress line a minute, the summary, the machine given back
from geolip.modular_diffusion import jobs, hub
PREFIX = {"stages": "modular_diffusion/stages", "chain_reads": "modular_diffusion/stages", "time_split": "modular_diffusion/time_split"}
RELEASE = True                              # give the machine back when the list has ended and the report is shipped
EXTRA = ""                                   # more arguments for every training job (empty in a real run)
EXTRA_READS = ""                             # more arguments for every chain read (empty in a real run)
kind = {}

def line(tag, exp, args, needs):
    kind[tag] = exp
    a = " " + args
    a += "" if " det=" in a else " det=on"
    a += "" if " device=" in a else " device=cuda threads=4"
    a += "" if " score=" in a else " score=1"
    a += " reader=1" if exp == "stages" and " score=0" not in a else ""      # the class read needs a scored job
    extra = EXTRA_READS if exp == "chain_reads" else EXTRA
    return f"{tag} | 60 | geolip.modular_diffusion.run | {exp}{a} data={NPZ} ref={REF}{extra}" + (f" | needs={','.join(needs)}" if needs else "")

tags = [t for t, *_ in ALL]
roster = f"{OUT}/roster.txt"
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(line(t, e, a, n) for t, e, a, n, _ in ALL) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    try:    # on a new machine: fetch this list's jobs that are already shipped and mark them done
        got_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX[kind[t]]}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
        for t in tags:
            src = f"{got_dir}/{PREFIX[kind[t]]}/{t}"
            if os.path.exists(f"{src}/result.json") and not mark(t, "DONE"):
                shutil.copytree(src, f"{RUNS}/{t}", dirs_exist_ok=True)
                open(f"{RUNS}/{t}/DONE", "w").write("fetched back from the Hub\n")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")

def after_job(tag, folder, ok):
    if ok and REPO:
        hub.ship(folder, REPO, f"{PREFIX[kind[tag]]}/{tag}", f"{kind[tag].replace('_', ' ')}: {tag}")

t0 = time.time()
left0 = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; {left0:,.0f} steps to run")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
runner.start()
while runner.is_alive():
    runner.join(60)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    eta = spent * left / (left0 - left) if left0 > left else float("nan")
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {eta:.0f} min left", flush=True)

summary = stages.summarize(RUNS) + "\n" + time_split.summarize(RUNS) + "\n" + chain_reads.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)   # the smoke's checks
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed (a job's own log files are not shipped with its folder)
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, "modular_diffusion/stages/_report_tree", "the stage tree and its parts: summary and roster logs") if REPO else None
if RELEASE and REPO and shipped:
    from google.colab import runtime
    runtime.unassign()